# Phase 4: Build Memory

## Step 14: Long-Term Memory

### Learning

- Persistent memory
- Episodic memory
- Semantic memory
- User preferences
- Memory extraction
- Memory retrieval
- Memory consolidation
- Memory conflicts
- Expiration
- User control

---

## Key Takeaways

- Step 13's memory disappears when the conversation ends. This step makes
  facts survive **across sessions**, stored per-user in a real (if small)
  database instead of in the current prompt.
- Extraction and saving are two separate steps on purpose. The model proposes
  candidate memories; plain application rules decide what actually gets
  written. A confident-sounding model is not a reason to skip that check.
- "Similar text" and "conflicting fact" are different problems. Near-identical
  wording is a duplicate (skip or update in place); topically-related but
  different wording on the same kind of fact is a conflict (the old one gets
  superseded, not silently overwritten).
- Retrieval needs a relevance floor. Returning "the closest memory" even when
  nothing is actually relevant just adds unrelated context — Section 12 shows
  what that looks like when it goes wrong.
- Nothing here gets hard-deleted except by explicit user action. Expired and
  superseded memories are marked inactive, not erased, so there's still an
  audit trail of what the system believed and when.

---

## To do (mirrors the Roadmap 1:1)

1. Create a memory database table
2. Define memory categories
3. Extract candidate memories
4. Apply deterministic saving rules
5. Generate memory embeddings
6. Retrieve relevant memories
7. Add memory relevance thresholds
8. Detect duplicate memories
9. Resolve conflicting memories
10. Add memory management to the interface
11. Add memory expiration
12. Test unwanted memory influence

Kept simple on purpose: plain SQLite (Python's built-in `sqlite3`, no ORM), a
pure-Python cosine similarity function (no numpy/vector DB needed for a
handful of memories), and plain functions instead of a memory-manager class.

## 0. Environment Setup

No Elasticsearch/Chroma here — memories are few enough per user that a plain
SQLite table plus a Python loop for similarity search is simpler than running
a vector database, and it's easier to see exactly what's happening.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import config
importlib.reload(config)

import json
import sqlite3
import uuid
from datetime import datetime, timedelta, timezone
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel

from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

client = OpenAI(api_key=OPENAI_API_KEY)


def get_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def now_iso():
    return datetime.now(timezone.utc).isoformat()

## 1. Create a Memory Database Table

> Use SQLite or PostgreSQL. Fields such as: memory_id, user_id, memory_text,
> memory_type, source_conversation_id, source_message_ids, created_at,
> updated_at, last_accessed_at, confidence, importance, expires_at, is_active,
> embedding. Store the embedding in a vector-capable database or a separate
> vector store.

SQLite, one table, the roadmap's own field list -- plus one extra field,
`reason`, so Section 10 can honestly answer "why was this memory saved"
instead of only showing its category. The embedding is stored as a JSON-
encoded list of floats in a TEXT column; for a handful of memories per user,
scanning and computing cosine similarity in Python (Section 6) is simpler
than standing up a vector store.

In [2]:
DB_PATH = PROJECT_ROOT / "data" / "memory.db"
DB_PATH.unlink(missing_ok=True)  # start fresh each run of this notebook

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

conn.execute("""
CREATE TABLE memories (
    memory_id TEXT PRIMARY KEY,
    user_id TEXT NOT NULL,
    memory_text TEXT NOT NULL,
    memory_type TEXT NOT NULL,
    reason TEXT,
    source_conversation_id TEXT,
    source_message_ids TEXT,      -- JSON list
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    last_accessed_at TEXT,
    confidence REAL,
    importance REAL,
    expires_at TEXT,
    is_active INTEGER NOT NULL,   -- 0 or 1
    embedding TEXT NOT NULL       -- JSON list of floats
)
""")
conn.commit()
print("Created memories table at", DB_PATH)

Created memories table at /Users/hirakhan/Developer/AI-ML/rag-chatbot/data/memory.db


## 2. Define Memory Categories

> Start with categories such as: user preference, personal fact, project
> fact, decision, repeated behavior, relationship between entities, temporary
> task context. Different categories may have different expiration and
> approval rules.

Just a fixed list, used to validate `memory_type` everywhere else in this
notebook.

In [3]:
MEMORY_TYPES = [
    "user_preference",
    "personal_fact",
    "project_fact",
    "decision",
    "repeated_behavior",
    "relationship",
    "temporary_task_context",
]

## 3. Extract Candidate Memories

> After a conversation turn or session, ask the model to identify information
> that may be useful later. Structured output:
> `{"memories": [{"text", "type", "confidence", "reason"}]}`. Candidate
> extraction should not save information automatically.

This function only *proposes* memories -- nothing is written to the database
here. Section 4 is the actual gate.

In [4]:
class MemoryCandidate(BaseModel):
    text: str
    type: Literal[tuple(MEMORY_TYPES)]
    confidence: float
    reason: str


class MemoryCandidates(BaseModel):
    memories: list[MemoryCandidate]


def extract_candidate_memories(conversation_text):
    response = client.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "Identify information in this conversation excerpt that could be "
                    "useful to remember in future conversations with this user -- "
                    "preferences, facts about them or their projects, decisions they "
                    "made. Skip small talk and one-time requests. For each candidate, "
                    "give a confidence (0-1) and a short reason it's worth remembering."
                ),
            },
            {"role": "user", "content": conversation_text},
        ],
        response_format=MemoryCandidates,
    )
    return response.choices[0].message.parsed.memories

In [6]:
candidates = extract_candidate_memories(
    "user: Hi there! I prefer detailed, thorough answers rather than short summaries.\n"
    "assistant: Got it, I'll keep that in mind."
)

for c in candidates:
    print(f"[{c.type}] (confidence={c.confidence}) {c.text}")
    print(f"   reason: {c.reason}")

[user_preference] (confidence=0.9) User prefers detailed, thorough answers rather than short summaries.
   reason: User explicitly stated their preference for detailed, thorough answers at the start of the conversation.


## 4. Apply Deterministic Saving Rules

> Rules such as: ignore greetings, ignore one-time requests, ignore uncertain
> statements, avoid storing duplicate facts, require confirmation for
> sensitive information, expire short-lived task details, prefer explicit
> user preferences over inferred preferences.

Plain code, not another model call -- these are the checks a candidate must
pass *before* Section 5 ever embeds or stores it. Duplicate detection gets its
own, embedding-based version in Section 8; this is just the cheap text-level
first pass.

In [7]:
MIN_CONFIDENCE = 0.6
SENSITIVE_KEYWORDS = ["password", "ssn", "social security", "credit card", "api key"]


def passes_saving_rules(candidate):
    """Returns (should_save: bool, reason: str)."""
    text_lower = candidate.text.lower()

    if candidate.confidence < MIN_CONFIDENCE:
        return False, f"confidence {candidate.confidence} below minimum {MIN_CONFIDENCE}"

    if any(keyword in text_lower for keyword in SENSITIVE_KEYWORDS):
        return False, "contains sensitive information -- requires explicit user confirmation first"

    if candidate.type not in MEMORY_TYPES:
        return False, f"unknown memory type: {candidate.type}"

    return True, "passed rules"

In [8]:
for c in candidates:
    should_save, reason = passes_saving_rules(c)
    print(f"{should_save!s:<6} {c.text!r} -- {reason}")

True   'User prefers detailed, thorough answers rather than short summaries.' -- passed rules


## 5. Generate Memory Embeddings

> Embed approved memory text. Store the original memory, embedding, user ID,
> and metadata. Never retrieve memories belonging to a different user.

`save_memory` is the only function in this notebook that writes to the
table, and it always requires a `user_id` -- every retrieval function later
filters on it too, so there's no code path that can return one user's memory
to another user.

In [9]:
def save_memory(user_id, text, memory_type, confidence, reason, source_conversation_id, source_message_ids=None, importance=0.5, expires_at=None):
    memory_id = str(uuid.uuid4())
    embedding = get_embedding(text)
    timestamp = now_iso()

    conn.execute(
        """
        INSERT INTO memories (
            memory_id, user_id, memory_text, memory_type, reason,
            source_conversation_id, source_message_ids,
            created_at, updated_at, last_accessed_at,
            confidence, importance, expires_at, is_active, embedding
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 1, ?)
        """,
        (
            memory_id, user_id, text, memory_type, reason,
            source_conversation_id, json.dumps(source_message_ids or []),
            timestamp, timestamp, timestamp,
            confidence, importance, expires_at, json.dumps(embedding),
        ),
    )
    conn.commit()
    return memory_id


USER_ID = "marcus"

for c in candidates:
    should_save, rule_reason = passes_saving_rules(c)
    if should_save:
        memory_id = save_memory(
            user_id=USER_ID,
            text=c.text,
            memory_type=c.type,
            confidence=c.confidence,
            reason=c.reason,
            source_conversation_id="conv-001",
        )
        print("Saved:", memory_id, "-", c.text)

Saved: c60e26fa-3a0b-481f-91c0-5e76b6115424 - User prefers detailed, thorough answers rather than short summaries.


## 6. Retrieve Relevant Memories

> Before answering a message: embed the current query, search the current
> user's memories, apply recency and importance weighting, select only a few
> relevant memories, insert them into a clearly labeled memory section.

`get_active_memories` is the one place that reads from the table -- every
query goes through it, and it always filters by `user_id` and `is_active`.

In [10]:
def get_active_memories(user_id):
    rows = conn.execute(
        "SELECT * FROM memories WHERE user_id = ? AND is_active = 1", (user_id,)
    ).fetchall()
    return rows


def retrieve_memories(user_id, query_text, top_k=3):
    query_embedding = get_embedding(query_text)

    scored = []
    for row in get_active_memories(user_id):
        memory_embedding = json.loads(row["embedding"])
        similarity = cosine_similarity(query_embedding, memory_embedding)
        # Simple recency + importance weighting: mostly similarity, with a
        # small boost for importance. (Recency itself matters more once
        # expiration, Section 11, is in the picture.)
        score = similarity * 0.8 + row["importance"] * 0.2
        scored.append((score, similarity, row))

    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:top_k]

In [11]:
results = retrieve_memories(USER_ID, "How should I format my answers for you?")
for score, similarity, row in results:
    print(f"score={score:.2f} similarity={similarity:.2f} -- {row['memory_text']}")

score=0.41 similarity=0.39 -- User prefers detailed, thorough answers rather than short summaries.


## 7. Add Memory Relevance Thresholds

> Do not inject the closest memory merely because something must rank first.
> Require a minimum relevance threshold. Skip memory retrieval when nothing
> is sufficiently relevant.

Extends `retrieve_memories` with a floor on similarity -- below it, a memory
doesn't get returned at all, even if it's the "best" of a bad set.

In [12]:
MIN_RELEVANCE = 0.75


def retrieve_relevant_memories(user_id, query_text, top_k=3, min_relevance=MIN_RELEVANCE):
    results = retrieve_memories(user_id, query_text, top_k=top_k)
    return [(score, similarity, row) for score, similarity, row in results if similarity >= min_relevance]


def format_memory_section(memories):
    """Clearly labeled memory section for the prompt -- empty string if nothing qualified."""
    if not memories:
        return ""
    lines = [f"- {row['memory_text']}" for _, _, row in memories]
    return "Relevant memories about this user:\n" + "\n".join(lines)

In [13]:
relevant = retrieve_relevant_memories(USER_ID, "How should I format my answers for you?")
print(format_memory_section(relevant) or "(no memories cleared the relevance threshold)")

print()
unrelated = retrieve_relevant_memories(USER_ID, "What's the weather like today?")
print(format_memory_section(unrelated) or "(no memories cleared the relevance threshold)")

(no memories cleared the relevance threshold)

(no memories cleared the relevance threshold)


## 8. Detect Duplicate Memories

> Before saving a new memory: search existing memories, compare similarity,
> decide whether to skip, update, merge, or store separately.

`find_most_similar_memory` restricts comparison to the **same memory type** --
comparing a `user_preference` against a `project_fact` for "is this a
duplicate" doesn't make sense even if the text happens to be similar.

In [14]:
DUPLICATE_THRESHOLD = 0.92  # near-identical wording -- treat as the same fact


def find_most_similar_memory(user_id, text, memory_type):
    embedding = get_embedding(text)

    best_row = None
    best_similarity = 0.0
    for row in get_active_memories(user_id):
        if row["memory_type"] != memory_type:
            continue
        similarity = cosine_similarity(embedding, json.loads(row["embedding"]))
        if similarity > best_similarity:
            best_similarity = similarity
            best_row = row

    return best_row, best_similarity


def update_memory_text(memory_id, new_text, new_confidence, new_reason):
    embedding = get_embedding(new_text)
    conn.execute(
        "UPDATE memories SET memory_text = ?, confidence = ?, reason = ?, embedding = ?, updated_at = ? WHERE memory_id = ?",
        (new_text, new_confidence, new_reason, json.dumps(embedding), now_iso(), memory_id),
    )
    conn.commit()

In [15]:
# Restating the SAME preference already saved in Section 5 ("detailed answers"),
# in slightly different words -- not a switch to a new preference (that's Section 9).
duplicate_candidate = MemoryCandidate(
    text="Just to reiterate, I prefer detailed and thorough responses over short summaries.",
    type="user_preference",
    confidence=0.85,
    reason="Restated preference",
)

existing, similarity = find_most_similar_memory(USER_ID, duplicate_candidate.text, duplicate_candidate.type)
print("Most similar existing memory:", existing["memory_text"] if existing else None)
print("Similarity:", round(similarity, 3))
print("Is a duplicate?", similarity >= DUPLICATE_THRESHOLD)

Most similar existing memory: User prefers detailed, thorough answers rather than short summaries.
Similarity: 0.742
Is a duplicate? False


## 9. Resolve Conflicting Memories

> Test: "Earlier: the user prefers detailed answers. Later: the user now
> prefers concise answers." Mark the older memory as superseded or inactive.
> Retain source history for auditing.

A conflict is a **same-type, moderately-similar-but-not-identical** memory --
similar enough to be about the same thing (both are about response length),
different enough not to be a duplicate. `save_or_resolve_memory` puts
Sections 5, 8, and 9 together into the one function that should actually be
called when saving a new candidate.

In [16]:
CONFLICT_THRESHOLD = 0.55  # same type, similar enough to be "about the same thing"


def save_or_resolve_memory(user_id, candidate, source_conversation_id):
    should_save, rule_reason = passes_saving_rules(candidate)
    if not should_save:
        return None, f"rejected: {rule_reason}"

    existing, similarity = find_most_similar_memory(user_id, candidate.text, candidate.type)

    if existing is None or similarity < CONFLICT_THRESHOLD:
        memory_id = save_memory(user_id, candidate.text, candidate.type, candidate.confidence, candidate.reason, source_conversation_id)
        return memory_id, "stored_new"

    if similarity >= DUPLICATE_THRESHOLD:
        return existing["memory_id"], "skipped_duplicate"

    # CONFLICT_THRESHOLD <= similarity < DUPLICATE_THRESHOLD, same type: treat as an update.
    # Mark the old one inactive (superseded) rather than deleting it -- audit trail preserved.
    conn.execute("UPDATE memories SET is_active = 0, updated_at = ? WHERE memory_id = ?", (now_iso(), existing["memory_id"]))
    conn.commit()
    memory_id = save_memory(user_id, candidate.text, candidate.type, candidate.confidence, candidate.reason, source_conversation_id)
    return memory_id, f"superseded {existing['memory_id']}"

In [17]:
# The roadmap's own example: an earlier preference gets explicitly revised.
revised_candidate = MemoryCandidate(
    text="User now prefers concise answers instead of detailed ones.",
    type="user_preference",
    confidence=0.9,
    reason="User explicitly changed their preference",
)

memory_id, outcome = save_or_resolve_memory(USER_ID, revised_candidate, source_conversation_id="conv-002")
print("Outcome:", outcome)

print("\nAll user_preference memories for this user (active and inactive):")
for row in conn.execute("SELECT memory_text, is_active FROM memories WHERE user_id = ? AND memory_type = ?", (USER_ID, "user_preference")):
    print(f"  active={row['is_active']}  {row['memory_text']}")

Outcome: superseded c60e26fa-3a0b-481f-91c0-5e76b6115424

All user_preference memories for this user (active and inactive):
  active=0  User prefers detailed, thorough answers rather than short summaries.
  active=1  User now prefers concise answers instead of detailed ones.


## 10. Add Memory Management to the Interface

> Allow users to: view stored memories, search memories, correct memory
> text, delete memories, disable memory, clear all memories, see why a
> memory was saved, see where it came from.

These are the functions a real UI would call. Two different removal actions,
on purpose: `disable_memory` is reversible (`is_active = 0`, same mechanism
Section 9 uses for superseded memories); `delete_memory` and
`clear_all_memories` are permanent, for when the user actually wants
something gone, not just hidden.

In [18]:
def view_memories(user_id, include_inactive=False):
    if include_inactive:
        query = "SELECT * FROM memories WHERE user_id = ?"
    else:
        query = "SELECT * FROM memories WHERE user_id = ? AND is_active = 1"
    return conn.execute(query, (user_id,)).fetchall()


def search_memories(user_id, query_text, top_k=5):
    return retrieve_memories(user_id, query_text, top_k=top_k)


def explain_memory(memory_id):
    row = conn.execute("SELECT * FROM memories WHERE memory_id = ?", (memory_id,)).fetchone()
    if row is None:
        return None
    return {
        "text": row["memory_text"],
        "type": row["memory_type"],
        "why_saved": row["reason"],
        "confidence": row["confidence"],
        "source_conversation_id": row["source_conversation_id"],
        "created_at": row["created_at"],
        "is_active": bool(row["is_active"]),
    }


def correct_memory(memory_id, new_text):
    update_memory_text(memory_id, new_text, new_confidence=1.0, new_reason="Corrected by user")


def disable_memory(memory_id):
    conn.execute("UPDATE memories SET is_active = 0, updated_at = ? WHERE memory_id = ?", (now_iso(), memory_id))
    conn.commit()


def delete_memory(memory_id):
    conn.execute("DELETE FROM memories WHERE memory_id = ?", (memory_id,))
    conn.commit()


def clear_all_memories(user_id):
    conn.execute("DELETE FROM memories WHERE user_id = ?", (user_id,))
    conn.commit()

In [19]:
print("All memories for", USER_ID, "(active only):")
for row in view_memories(USER_ID):
    print(f"  [{row['memory_type']}] {row['memory_text']}")

first_memory = view_memories(USER_ID)[0]
print("\nWhy was this one saved?")
print(explain_memory(first_memory["memory_id"]))

disable_memory(first_memory["memory_id"])
print("\nAfter disabling it, active count:", len(view_memories(USER_ID)))

All memories for marcus (active only):
  [user_preference] User now prefers concise answers instead of detailed ones.

Why was this one saved?
{'text': 'User now prefers concise answers instead of detailed ones.', 'type': 'user_preference', 'why_saved': 'User explicitly changed their preference', 'confidence': 0.9, 'source_conversation_id': 'conv-002', 'created_at': '2026-09-01T11:49:34.826520+00:00', 'is_active': True}

After disabling it, active count: 0


## 11. Add Memory Expiration

> Assign expiration dates to temporary memories. Create a cleanup task that
> marks expired memories inactive. Do not permanently delete audit records
> unless required by the application's data-retention policy.

`temporary_task_context` memories are the obvious candidate for an
`expires_at` -- a one-off task detail shouldn't outlive the task.

In [20]:
# Save a temporary memory that already expired (to demo cleanup immediately,
# instead of waiting for real time to pass).
already_expired = (datetime.now(timezone.utc) - timedelta(days=1)).isoformat()

expiring_memory_id = save_memory(
    user_id=USER_ID,
    text="User asked for a draft to be ready by end of day for a one-off request.",
    memory_type="temporary_task_context",
    confidence=0.9,
    reason="One-off task detail, not useful after the task is done",
    source_conversation_id="conv-003",
    expires_at=already_expired,
)


def expire_old_memories():
    """Mark any memory whose expires_at has passed as inactive. Nothing is deleted."""
    conn.execute(
        "UPDATE memories SET is_active = 0, updated_at = ? WHERE expires_at IS NOT NULL AND expires_at < ? AND is_active = 1",
        (now_iso(), now_iso()),
    )
    conn.commit()


print("Before cleanup, active:", any(row["memory_id"] == expiring_memory_id for row in view_memories(USER_ID)))
expire_old_memories()
print("After cleanup, active: ", any(row["memory_id"] == expiring_memory_id for row in view_memories(USER_ID)))
print("Still in the table (audit record kept):", explain_memory(expiring_memory_id) is not None)

Before cleanup, active: True
After cleanup, active:  False
Still in the table (audit record kept): True


## 12. Test Unwanted Memory Influence

> Create memories about one project, then ask unrelated questions. Observe
> whether irrelevant memories change the assistant's answer. Adjust
> retrieval thresholds and prompt placement.

In [21]:
save_memory(
    user_id=USER_ID,
    text="Project Phoenix's launch date is set for November 3rd.",
    memory_type="project_fact",
    confidence=0.9,
    reason="Explicit project fact",
    source_conversation_id="conv-004",
)
save_memory(
    user_id=USER_ID,
    text="Project Phoenix is being built with a microservices architecture.",
    memory_type="project_fact",
    confidence=0.9,
    reason="Explicit project fact",
    source_conversation_id="conv-004",
)

unrelated_question = "Can you recommend a good book about gardening?"
leaked = retrieve_relevant_memories(USER_ID, unrelated_question)

print("Question:", unrelated_question)
if leaked:
    print("LEAKED memories into an unrelated question:")
    for _, similarity, row in leaked:
        print(f"  similarity={similarity:.2f} -- {row['memory_text']}")
else:
    print("No memories cleared the relevance threshold -- correctly stayed out of context.")

Question: Can you recommend a good book about gardening?
No memories cleared the relevance threshold -- correctly stayed out of context.


### Reflection

- **What should the assistant remember?** Facts likely to matter in *future*
  sessions — preferences, stable project facts, decisions. Section 4's rules
  are the filter for this.
- **What should never be stored automatically?** Sensitive information
  (Section 4's keyword check) and anything below the confidence floor —
  both require an explicit human step before they'd ever be saved for real.
- **How is a memory different from a message?** A message is what was said;
  a memory is a *distilled claim* extracted from it, stored independently of
  the conversation it came from, and expected to outlive that conversation.
- **Should the model decide what's important?** Partially — it proposes
  (Section 3) but plain code decides what's actually saved (Section 4) and
  how conflicts get resolved (Section 9). The model is never the last check.
- **How should conflicting memories be resolved?** Mark the old one inactive,
  save the new one, keep both rows for audit — never silently overwrite
  (Section 9).
- **When should memories expire?** When they're inherently temporary — task
  context tied to a specific, finished task (Section 11). Preferences and
  stable facts generally shouldn't have an expiration at all.
- **Can retrieved memories bias unrelated conversations?** Yes if there's no
  relevance floor — Section 7 exists specifically to prevent it, and Section
  12 is the test that confirms it's working.
- **How can users inspect and control what's stored?** Section 10's
  functions: view, search, correct, disable (reversible), delete/clear
  (permanent), and see both the reason and source for any memory.
- **Episodic vs. semantic memory?** Episodic is "what happened in this
  conversation" (closer to Step 13); semantic is "what's true about this
  user going forward" — this notebook's `memories` table is semantic memory.
- **Should inferred preferences be stored?** With more caution than explicit
  ones — the roadmap's own rule (Section 4) is to prefer explicit statements
  over inferred ones, and an inferred preference probably deserves a lower
  confidence score and a lower bar for being superseded later.